# 🕶️ Detector de Óculos Escuros em Fotos de Candidatos (Google Colab)

Este notebook executa a detecção híbrida de **óculos escuros (sunglasses)** utilizando **YOLO-World + CLIP Zero-Shot Classification** com limiar de **confiança estrito superior a 90% (>0.90)**.

Garante a **eliminação total de falsos positivos**, diferenciando óculos de grau transparentes de óculos escuros de sol.

---

## 🛠️ Passo 1: Instalação das Bibliotecas Necessárias no Google Colab

In [ ]:
# Instalar as dependências do YOLO-World, CLIP da Ultralytics e Transformers
!pip install -q ultralytics transformers pillow pandas ftfy git+https://github.com/ultralytics/CLIP.git

## 📁 Passo 2: Extrair Fotos (Amostra do GitHub ou foto_cand2024_SP_div.zip do Drive)

Escolha uma das opções abaixo no Colab. A **Opção B** detecta automaticamente o arquivo `foto_cand2024_SP_div.zip` na pasta do seu Google Drive e descompacta no SSD local.

In [ ]:
# OPÇÃO A: Extrair fotos de amostra diretamente do GitHub (Recomendado para testes rápidos)
import os
import shutil

!git clone https://github.com/koiti/fotos_candidatos.git

if os.path.exists('fotos_candidatos/amostras'):
    if os.path.exists('amostras'):
        shutil.rmtree('amostras')
    shutil.copytree('fotos_candidatos/amostras', 'amostras')
    print("-> Fotos de amostra extraídas com sucesso do GitHub para a pasta 'amostras'!")

In [ ]:
# OPÇÃO B: Processar Dataset Grande (foto_cand2024_SP_div.zip com 78k fotos e 2GB) via Google Drive
# DICA DE DESEMPENHO: O código busca o arquivo .zip no seu Drive (na mesma pasta do notebook ou subpastas),
# copia para o SSD local da máquina do Colab em segundos e descompacta lá. Isso deixa o I/O 50x mais rápido!
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

zip_name = 'foto_cand2024_SP_div.zip'
zip_found = None

# Verificar locais comuns no Google Drive
search_paths = [
    Path('/content/drive/MyDrive') / zip_name,
    Path('/content/drive/MyDrive/UFABC/2026-TOPICOS_DE_IA') / zip_name,
    Path('/content/drive/MyDrive/2026-TOPICOS_DE_IA') / zip_name
]

for p in search_paths:
    if p.exists():
        zip_found = p
        break

if not zip_found:
    print(f"🔍 Procurando '{zip_name}' no seu Google Drive...")
    matches = list(Path('/content/drive/MyDrive').rglob(zip_name))
    if matches:
        zip_found = matches[0]

if zip_found:
    print(f"-> Arquivo localizado com sucesso: {zip_found}")
    print("-> Copiando .zip do Drive para o SSD ultrarrápido do Colab...")
    !cp "{zip_found}" /content/
    print("-> Descompactando 78.000 fotos no SSD local...")
    !unzip -q /content/foto_cand2024_SP_div.zip -d /content/foto_cand2024_SP
    print("-> SUCESSO! 78.000 fotos prontas para análise ultrarrápida na pasta '/content/foto_cand2024_SP'!")
else:
    print(f"⚠️ Arquivo '{zip_name}' não encontrado no seu Google Drive. Verifique se o upload foi concluído.")

In [ ]:
# OPÇÃO C: Fazer upload direto de um arquivo ZIP contendo as fotos (ex: amostras.zip)
from google.colab import files
import zipfile

print('Faça o upload do seu arquivo .zip com as fotos:')
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('amostras')
        print("-> Arquivos extraídos com sucesso na pasta 'amostras'!")

## 🧠 Passo 3: Código do Detector Híbrido de Óculos Escuros (YOLO + CLIP)

In [ ]:
import os
os.environ['YOLO_AUTOINSTALL'] = 'False'

import json
import time
import torch
from pathlib import Path
from datetime import datetime
from PIL import Image, ImageDraw, ImageOps
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel
from IPython.display import display, Image as IPImage
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de inferência no Colab: {device.upper()}")

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_PROMPTS = [
    "dark black tinted sunglasses covering eyes",
    "clear transparent prescription reading eyeglasses with visible eyes"
]

def executar_deteccao_oculos_escuros_colab(
    input_dir="amostras",
    output_dir="irregular_oculos_escuros",
    clip_thresh=0.90,
    batch_size=64
):
    target_input = Path(input_dir)
    if not target_input.exists() and Path('/content/foto_cand2024_SP').exists():
        target_input = Path('/content/foto_cand2024_SP')

    target_output = Path(output_dir)
    target_output.mkdir(exist_ok=True, parents=True)

    fotos = sorted([
        f for f in target_input.iterdir()
        if f.is_file() and f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    ])

    if not fotos:
        print(f"Nenhuma foto encontrada na pasta '{target_input}'!")
        return None

    print(f"Analizando {len(fotos)} fotos em '{target_input}' com limiar CLIP de {clip_thresh:.0%}...")
    print("1. Carregando YOLO-World para localização de armações...")
    model_yolo = YOLO("yolov8s-worldv2.pt")
    model_yolo.set_classes(["glasses", "eyeglasses", "sunglasses"])

    print("2. Carregando modelo CLIP para verificação semântica das lentes...")
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
    print("Modelos inicializados com sucesso!")

    total_irregulares = 0
    resultados_detalhados = []
    t_start = time.time()

    for b_idx in range(0, len(fotos), batch_size):
        bfiles = fotos[b_idx:b_idx + batch_size]
        bimgs = []
        vfiles = []
        for f in bfiles:
            try:
                img = Image.open(f)
                img = ImageOps.exif_transpose(img).convert("RGB")
                bimgs.append(img)
                vfiles.append(f)
            except Exception:
                pass

        if not bimgs:
            continue

        results = model_yolo.predict(source=bimgs, conf=0.20, batch=batch_size, device=device, verbose=False)

        for foto_path, img_rgb, res in zip(vfiles, bimgs, results):
            irregularidades = []
            if len(res.boxes) > 0:
                boxes = res.boxes.xyxy.cpu().numpy()
                w, h = img_rgb.size

                for box in boxes:
                    x1, y1, x2, y2 = [int(c) for c in box]
                    bw, bh = x2 - x1, y2 - y1
                    cx1 = max(0, int(x1 - bw * 0.1))
                    cy1 = max(0, int(y1 - bh * 0.1))
                    cx2 = min(w, int(x2 + bw * 0.1))
                    cy2 = min(h, int(y2 + bh * 0.1))

                    crop = img_rgb.crop((cx1, cy1, cx2, cy2))
                    inputs_clip = clip_processor(text=CLIP_PROMPTS, images=crop, return_tensors="pt", padding=True).to(device)
                    with torch.no_grad():
                        outputs_clip = clip_model(**inputs_clip)
                        probs = outputs_clip.logits_per_image.softmax(dim=-1)[0]

                    prob_dark = float(probs[0])
                    prob_clear = float(probs[1])

                    if prob_dark >= clip_thresh and prob_dark > prob_clear:
                        irregularidades.append({
                            "classe_en": "dark sunglasses",
                            "classe_pt": "óculos escuros",
                            "confianca_clip": round(prob_dark, 4),
                            "prob_oculos_grau": round(prob_clear, 4),
                            "bbox": [x1, y1, x2, y2]
                        })

            if irregularidades:
                total_irregulares += 1
                draw = ImageDraw.Draw(img_rgb)
                for det in irregularidades:
                    box = det["bbox"]
                    label_pt = det["classe_pt"]
                    conf = det["confianca_clip"]
                    draw.rectangle(box, outline="red", width=4)
                    draw.text((box[0] + 5, max(0, box[1] - 15)), f"{label_pt} ({conf:.1%})", fill="red")
                img_rgb.save(target_output / foto_path.name)
                print(f" -> DETECTADO: {foto_path.name} | Óculos Escuros ({irregularidades[0]['confianca_clip']:.1%})")

            resultados_detalhados.append({
                "arquivo": foto_path.name,
                "status": "irregular" if irregularidades else "regular",
                "irregularidades": irregularidades
            })

    t_total = time.time() - t_start
    relatorio_data = {
        "data_analise": datetime.now().isoformat(),
        "modelo": "YOLO-World + CLIP",
        "limiar_clip": clip_thresh,
        "total_fotos": len(fotos),
        "total_irregulares": total_irregulares,
        "tempo_execucao_segundos": round(t_total, 2),
        "resultados": resultados_detalhados
    }

    relatorio_path = target_output / "relatorio.json"
    with open(relatorio_path, "w", encoding="utf-8") as f:
        json.dump(relatorio_data, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 70)
    print(f"VARREDURA CONCLUÍDA NO COLAB!")
    print(f"Total de fotos analisadas: {len(fotos)}")
    print(f"Irregulares salvas em '{target_output}': {total_irregulares}")
    print(f"Relatório salvo em: {relatorio_path}")
    return relatorio_data

## 🚀 Passo 4: Executar a Detecção

In [ ]:
# Executar a detecção com limiar de 90% de certeza no CLIP
# Por padrão usa 'amostras' se existir, ou '/content/foto_cand2024_SP' se o dataset do Drive foi extraído
pasta_fotos = '/content/foto_cand2024_SP' if os.path.exists('/content/foto_cand2024_SP') else 'amostras'

resultado = executar_deteccao_oculos_escuros_colab(
    input_dir=pasta_fotos,
    output_dir="irregular_oculos_escuros",
    clip_thresh=0.90
)

# Exibir dataframe com resultados irregulares
if resultado:
    df = pd.DataFrame(resultado["resultados"])
    display(df[df["status"] == "irregular"])

## 🖼️ Passo 5: Visualizar Imagens Irregulares no Colab

In [ ]:
# Exibir fotos anotadas com bounding box vermelha
target = Path("irregular_oculos_escuros")
fotos_irregulares = sorted([f for f in target.glob("*.[jJ][pP]*[gG]")])

if fotos_irregulares:
    print(f"Exibindo {len(fotos_irregulares)} fotos irregulares:")
    for f in fotos_irregulares[:20]:  # Exibe até as primeiras 20 para visualização
        print(f"📷 {f.name}")
        display(IPImage(filename=str(f), width=350))
else:
    print("Nenhuma imagem irregular encontrada com mais de 90% de certeza!")

## 💾 Passo 6: Baixar Resultados (.zip)

In [ ]:
# Compactar a pasta de resultados e fazer download para seu computador
!zip -r irregular_oculos_escuros.zip irregular_oculos_escuros
from google.colab import files
files.download('irregular_oculos_escuros.zip')